In [1]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error


In [38]:
X_train_encoded = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_train_encoded.parquet").drop(columns=['Date'])
X_valid_encoded = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_valid_encoded.parquet").drop(columns=['Date'])
X_predict_encoded = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_predict_encoded.parquet").drop(columns=['Date'])
y_train = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\y_train.parquet").drop(columns=['Date'])
y_valid = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\y_valid.parquet").drop(columns=['Date'])
y_predict = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\y_predict.parquet").drop(columns=['Date'])

In [3]:
y_train_home = y_train["FTHG"]
y_valid_home = y_valid["FTHG"]
y_predict_home = y_predict["FTHG"]

y_train_away = y_train["FTAG"]
y_valid_away = y_valid["FTAG"]
y_predict_away = y_predict["FTAG"]

In [4]:
xgb = XGBRegressor(random_state=67, n_jobs=-1)
parameters = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5, 6],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

TimeSeriesSplit = TimeSeriesSplit(n_splits=5)

RandomizedSearchCV = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=parameters,
    n_iter=50,
    cv=TimeSeriesSplit,
    scoring='neg_mean_absolute_error',
    random_state=67,
    n_jobs=-1,
    verbose=1
)

In [5]:
RandomizedSearchCV.fit(X_train_encoded, y_train_home)
xgb_home_best = RandomizedSearchCV.best_estimator_
print(f"Best parameters: {RandomizedSearchCV.best_params_}")
print(f"Best CV MAE: {(-1) * RandomizedSearchCV.best_score_}")

RandomizedSearchCV.fit(X_train_encoded, y_train_away)
xgb_away_best = RandomizedSearchCV.best_estimator_
print(f"Best parameters: {RandomizedSearchCV.best_params_}")
print(f"Best CV MAE: {(-1) * RandomizedSearchCV.best_score_}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters: {'subsample': 0.8, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
Best CV MAE: 0.9903651999700689
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters: {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
Best CV MAE: 0.8993008753785162


In [16]:
home_pred = xgb_home_best.predict(X_valid_encoded)
home_mae = mean_absolute_error(y_valid_home,home_pred)
home_rmse = root_mean_squared_error(y_valid_home,home_pred)
print("FTHG Validation MAE:", home_mae)
print("FTHG Validation RMSE:", home_rmse)

away_pred = xgb_away_best.predict(X_valid_encoded)
away_mae = mean_absolute_error(y_valid_home,away_pred)
away_rmse = root_mean_squared_error(y_valid_home,away_pred)
print("FTAG Validation MAE:", away_mae)
print("FTAG Validation RMSE:", away_rmse)

FTHG Validation MAE: 0.9624115623925862
FTHG Validation RMSE: 1.141439189882753
FTAG Validation MAE: 1.034254062803168
FTAG Validation RMSE: 1.2822872648917032


In [17]:
print("FTHG parameters:")
print({
    "n_estimators": xgb_home_best.get_params()["n_estimators"],
    "learning_rate": xgb_home_best.get_params()["learning_rate"],
    "max_depth": xgb_home_best.get_params()["max_depth"],
    "min_child_weight": xgb_home_best.get_params()["min_child_weight"],
    "subsample": xgb_home_best.get_params()["subsample"],
    "colsample_bytree": xgb_home_best.get_params()["colsample_bytree"]
})

print("\nFTAG parameters:")
print({
    "n_estimators": xgb_away_best.get_params()["n_estimators"],
    "learning_rate": xgb_away_best.get_params()["learning_rate"],
    "max_depth": xgb_away_best.get_params()["max_depth"],
    "min_child_weight": xgb_away_best.get_params()["min_child_weight"],
    "subsample": xgb_away_best.get_params()["subsample"],
    "colsample_bytree": xgb_away_best.get_params()["colsample_bytree"]
})

FTHG parameters:
{'n_estimators': 200, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 3, 'subsample': 0.8, 'colsample_bytree': 0.8}

FTAG parameters:
{'n_estimators': 500, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 3, 'subsample': 0.8, 'colsample_bytree': 1.0}


In [32]:
X_train = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_train.parquet")
X_valid = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_valid.parquet")

In [43]:
pd.DataFrame({
    'Date': X_valid['Date'].values,
    'HomeTeam': X_valid['HomeTeam'].values,
    'AwayTeam': X_valid['AwayTeam'].values,
    'Actual_Score': (
        y_valid['FTHG'].astype(str) + '-' + y_valid['FTAG'].astype(str)
    ).values,
    'Pred_Score': (
        home_pred.round(0).astype(int).astype(str) + '-' + away_pred.round(0).astype(int).astype(str)
    )
}).sample(50)

,Date,HomeTeam,AwayTeam,Actual_Score,Pred_Score
29,2025-08-31,Nott'm Forest,West Ham,0.0-3.0,2-1
63,2025-10-04,Man United,Sunderland,2.0-0.0,2-1
378,2026-05-24,Tottenham,Everton,1.0-0.0,2-1
219,2026-01-19,Brighton,Bournemouth,1.0-1.0,2-1
241,2026-02-07,Bournemouth,Aston Villa,1.0-1.0,2-1
252,2026-02-10,Everton,Bournemouth,1.0-2.0,2-1
20,2025-08-30,Tottenham,Bournemouth,0.0-1.0,2-1
372,2026-05-24,Brighton,Man United,0.0-3.0,1-1
48,2025-09-21,Sunderland,Aston Villa,1.0-1.0,1-1
33,2025-09-13,Newcastle,Wolves,1.0-0.0,2-1


In [25]:
pd.DataFrame({
    'HomeTeam': X_valid['HomeTeam'].values,
    'AwayTeam': X_valid['AwayTeam'].values,
    'Actual_FTHG': y_valid['FTHG'].values,
    'Actual_FTAG': y_valid['FTAG'].values,
    'Pred_FTHG': home_pred.round(2),
    'Pred_FTAG': away_pred.round(2)
}).describe()

,Actual_FTHG,Actual_FTAG,Pred_FTHG,Pred_FTAG
count,380.000000,380.000000,380.000000,380.000000
mean,1.526316,1.223684,1.545079,1.258789
std,1.169914,1.084779,0.389756,0.276647
min,0.000000,0.000000,0.580000,0.750000
25%,1.000000,0.000000,1.310000,1.070000
50%,1.000000,1.000000,1.480000,1.210000
75%,2.000000,2.000000,1.710000,1.410000
max,5.000000,5.000000,3.220000,2.310000
